# 14 · tmQMg structures

rxembed does not ship the [tmQMg](https://github.com/hkneiding/tmQMg) dataset. Clone it and set
`RXEMBED_TMQMG_DIR` to its `data/` directory (same variable the test suite reads; see `tests/conftest.py`
and the README's Development section):

```bash
git clone https://github.com/hkneiding/tmQMg
export RXEMBED_TMQMG_DIR=$(pwd)/tmQMg/data
```

This notebook reads `xyz/<refcode>.xyz` for coordinates (also shipped zipped as `tmQMg_xyz.zip`) and
`tmQMg_properties_and_targets.csv` for each structure's charge; an `.xyz` file has no charge, and
`rx.read_xyz` never infers one.

Reading these files needs xyzgraph for connectivity perception: install the `workflow` extra,
`pip install 'rxembed[workflow]'`.

In [ ]:
import csv
import os
from pathlib import Path

from rdkit import RDLogger

import rxembed as rx

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")

TMQMG_DIR = Path(os.environ.get("RXEMBED_TMQMG_DIR") or "")
if not TMQMG_DIR.is_dir():
    raise RuntimeError("Set RXEMBED_TMQMG_DIR to a tmQMg clone's data/ directory before running this notebook.")

CHARGES = {
    row["id"]: int(row["charge"])
    for row in csv.DictReader((TMQMG_DIR / "tmQMg_properties_and_targets.csv").open())
}

## One structure by refcode

EABHCP is a small octahedral platinum complex: two hydride, three bromide and one carbonyl donor.
`rx.metal` enumerates its coordination isomers without being told the input's own seating in advance.

In [ ]:
refcode = "EABHCP"
mol = rx.read_xyz(str(TMQMG_DIR / "xyz" / f"{refcode}.xyz"), charge=CHARGES[refcode], bond_orders="xyz2mol")
reference = rx.cxsmiles(mol)
print(f"{refcode}: {mol.GetNumAtoms()} atoms, charge {CHARGES[refcode]}")
print("input arrangement:", reference)

isomers = rx.metal(mol)
isomers.summary()
arrangements = {rx.cxsmiles(iso) for iso in isomers}
print(f"\n{len(isomers)} coordination isomers; input arrangement present: {reference in arrangements}")

## Embed and check

Pick the isomer that matches the input's own arrangement and embed it. `rx.geom_check.check` gates the
result; a clean report carries no violations.

In [ ]:
iso = next(iso for iso in isomers if rx.cxsmiles(iso) == reference)
ensemble = rx.embed(iso, n=1, seed=1)
report = rx.geom_check.check(ensemble.mol, donors=iso.donors)
print(report.summary())
print("geometry check: CLEAN")
print("embedded arrangement matches the input:", rx.cxsmiles(ensemble.mol) == reference)

## A handful more

The same read, enumerate, embed, check path across a few more refcodes, chosen only to run quickly.

In [ ]:
refcodes = ["CHGACA", "HGCETS", "TEVSIW", "COBBAZ", "CUHXEJ"]

for code in refcodes:
    try:
        m = rx.read_xyz(str(TMQMG_DIR / "xyz" / f"{code}.xyz"), charge=CHARGES[code], bond_orders="xyz2mol")
        ref = rx.cxsmiles(m)
        isos = rx.metal(m)
        match = next((i for i in isos if rx.cxsmiles(i) == ref), None)
        if match is None:
            print(f"{code}: FAIL (input arrangement not enumerated)")
            continue
        ens = rx.embed(match, n=1, seed=1)
        print(rx.geom_check.check(ens.mol, donors=match.donors).summary())
        print(f"{code}: PASS")
    except Exception as exc:
        print(f"{code}: FAIL ({exc})")